# 10장 스무고개와 집단지성: 결정나무와 앙상블 (실습 v8)

앞쪽은 01~06절의 작은 [실습 예제](은행 데이터 없이 몇 줄로 개념 확인), 뒤쪽은 05절 [실습 예제 10-1]~[10-9]의 단계(❶❷❸…)입니다. 05절 실습은 앞 단계의 변수를 이어 쓰므로 위에서부터 차례로 실행하세요. '직접 해 보기' 셀은 책의 [직접 해 보기] 정답입니다. 🔵 코랩 실습 표시가 붙은 10개는 책에 문제만 실린 실습입니다.

In [ ]:
!pip install -q koreanize-matplotlib

## 준비: 라이브러리와 한글 글꼴

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 100)

# 01~06절 작은 실습

## [실습 예제 1.1] 고객 10명으로 나무 키우기

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
import pandas as pd
leaf = pd.DataFrame({"잎": ["지난번 가입·주택대출 없음", "지난번 가입·주택대출 있음", "지난번 가입 안 함"],
                     "고객": [1040, 471, 43700],          # 잎에 모인 고객 수
                     "가입률": [0.701, 0.529, 0.099]})    # 잎 안의 가입 비율 = 판정 확률
leaf["100명 걸면"] = (leaf["가입률"] * 100).round()       # 100명에게 걸 때 만날 가입자
print(leaf)
print("가 / 나 =", round(0.701 / 0.099, 1), "배")

### ❷ 넓혀 보기

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
d = pd.DataFrame({                          # 고객 10명의 작은 표
    "prev": [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],   # 지난번 가입(1) / 안 함(0)
    "loan": [0, 0, 1, 1, 0, 0, 1, 1, 1, 0],   # 대출 있음(1) / 없음(0)
    "y":    [1, 1, 1, 0, 1, 0, 0, 0, 0, 0]})  # 이번에 가입(1) / 안 함(0)
t = DecisionTreeClassifier(max_depth=2, random_state=0)   # 질문 두 번까지
t.fit(d[["prev", "loan"]], d["y"])          # 문제(두 열)와 답(y)으로 나무 키우기
new = pd.DataFrame({"prev": [1, 1, 0], "loan": [0, 1, 0]})  # 새 고객 셋
new["p"] = t.predict_proba(new)[:, 1]       # 가입 확률(둘째 열)
print(new.round(2))

## [실습 예제 1.2] 문턱을 모두 시험하기 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

In [ ]:
import numpy as np
age = np.array([25, 33, 41, 52, 63, 70])   # 고객 6명의 나이
y = np.array([0, 0, 0, 1, 1, 0])           # 가입 여부
def gini(v):                               # 0·1 배열의 지니 불순도
    p = v.mean()                           # 가입 비율
    return 2 * p * (1 - p)                 # = 1 - p² - (1-p)²
print("나누기 전", round(gini(y), 3))
for c in (age[:-1] + age[1:]) / 2:         # 이웃한 두 나이의 가운데 = 문턱 후보
    L, R = y[age <= c], y[age > c]         # 문턱 아래 / 위 두 무리
    w = (len(L) * gini(L) + len(R) * gini(R)) / len(y)   # 사람 수로 가중 평균
    print(f"{c:4.1f}세 문턱: 나눈 뒤 {w:.3f}")

## [실습 예제 1.3] 외우는 나무 만들어 보기

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
from sklearn.metrics import roc_auc_score
y = [1, 1, 0, 0, 0]                        # 가입 2명, 안 함 3명
p = [0.8, 0.4, 0.6, 0.3, 0.2]              # 모델이 준 가입 확률
pairs = [(a, b) for a, ya in zip(p, y) for b, yb in zip(p, y) if ya == 1 and yb == 0]
win = sum(a > b for a, b in pairs)         # 가입한 쪽 확률이 더 높은 짝
print("짝", len(pairs), "/ 이긴 짝", win, "/ AUC", round(win / len(pairs), 3))
print("roc_auc_score", round(roc_auc_score(y, p), 3))

### ❷ 넓혀 보기

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
rng = np.random.default_rng(0)             # 같은 난수가 나오게 씨앗 고정
X = rng.normal(size=(800, 5))              # 변수 5개, 800명(가짜 데이터)
y = X[:, 0] + rng.normal(0, 1.5, 800) > 1.2   # 첫 변수 + 잡음으로 정한 답
print("찍기(모두 아니요)", (~y[600:]).mean())
for d in [2, 5, None]:                     # None = 끝까지 자라게
    t = DecisionTreeClassifier(max_depth=d, random_state=0)
    t.fit(X[:600], y[:600])                # 앞 600명으로 훈련
    print(d, "훈련", round(t.score(X[:600], y[:600]), 3),
          "/ 시험", round(t.score(X[600:], y[600:]), 3))   # 뒤 200명으로 시험

## [실습 예제 1.4] 다시 뽑으면 몇 %가 들어갈까 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
import numpy as np
guess = np.array([1050, 1150, 1210, 1260, 1330])   # 다섯 사람의 어림
true = 1198                                         # 실제 무게
print("한 사람씩 오차", np.abs(guess - true), "평균", np.abs(guess - true).mean())
print("평균", guess.mean(), "오차", abs(guess.mean() - true))
print("한가운데 값", np.median(guess), "오차", abs(np.median(guess) - true))

### ❷ 넓혀 보기

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
for n in [10, 1000, 33908]:                # 원래 기록의 수
    idx = rng.integers(0, n, size=n)       # 0~n-1 번호를 n번, 중복을 허락해 뽑기
    u = len(np.unique(idx)) / n            # 한 번이라도 뽑힌 기록의 비율
    print(n, round(u, 3))
n = 33908
print("식 1-(1-1/n)^n", round(1 - (1 - 1 / n) ** n, 3))

## [실습 예제 1.5] 잔차를 따라 세 걸음 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
import numpy as np
age = np.array([35, 48, 58, 62, 71])       # 나이(세)
month = age * 12                           # 개월로 바꾸기
print("개월", month)
print("60세 초과   ", age > 60)
print("720개월 초과", month > 720)          # 같은 사람이 같은 쪽으로 간다

### ❷ 넓혀 보기

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
x = np.arange(1, 7).reshape(-1, 1)         # 고객 6명(번호를 특징으로)
y = np.array([0, 0, 1, 0, 1, 1.0])         # 가입 여부
F = np.full(6, y.mean())                   # 출발 예측: 모두 0.5
for m in range(1, 4):                      # 나무 세 그루를 차례로
    h = DecisionTreeRegressor(max_depth=1).fit(x, y - F)   # 잔차(y - F)를 맞히는 작은 나무
    F = F + 0.5 * h.predict(x)             # 학습률 0.5만큼만 더한다
    print(m, F.round(2), "잔차 제곱합", round(((y - F) ** 2).sum(), 2))

## [실습 예제 2.3] 작은 차이가 커지는 예보 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
import numpy as np
rain = np.array([0, 0, 0.5, 2, 3, 0, 4, 1.5, 0, 6])   # 예보 10개의 강수량(mm)
print("비(1mm 이상) 예보", (rain >= 1).sum(), "개 → 비 올 확률", (rain >= 1).mean())
print("평균", rain.mean(), "mm / 가장 적게", rain.min(), "/ 가장 많이", rain.max())

### ❷ 넓혀 보기

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = 0.4 + rng.normal(0, 0.001, 51)         # 출발값을 아주 조금씩 바꾼 예보 51개
x[0] = 0.4                                 # 0번은 바꾸지 않은 예보
for day in range(1, 31):
    x = 3.9 * x * (1 - x)                  # 하루치 계산(작은 차이가 커지는 식)
    if day in (5, 15, 30):
        print(day, "일째 흩어짐", round(x.std(), 3),
              "/ 0.5 넘는 예보 비율", (x > 0.5).mean().round(2))

## [실습 예제 3.1] 달의 차이인가, 해의 차이인가 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
import numpy as np
m = np.array([11, 12, 12, 1, 2, 5, 11, 1, 3])   # 날짜순 기록 9건의 달
year = 2008 + np.r_[0, (m[1:] < m[:-1]).cumsum()]   # 달이 거꾸로 가면 다음 해
print(year)
print({int(y): int((year == y).sum()) for y in np.unique(year)}, "/ 마지막", year[-1], "년", m[-1], "월")

### ❷ 넓혀 보기

In [ ]:
import pandas as pd
d = pd.DataFrame({"year":  [2008, 2009, 2009, 2010],
                  "month": ["5월", "3월", "5월", "3월"],
                  "n":     [800, 150, 500, 100],      # 연락한 사람
                  "join":  [40, 30, 100, 50]})        # 가입한 사람
for k in ["month", "year"]:                # 달별로, 해별로 따로 묶기
    s = d.groupby(k)[["n", "join"]].sum()  # 묶음마다 합
    print(k, (s["join"] / s["n"]).round(3).to_dict())   # 가입률
d["rate"] = (d["join"] / d["n"]).round(3)  # 해 안에서 달끼리 견주기
print(d)

## [실습 예제 3.2] 섞어 나누기와 시간 분할 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
import numpy as np
y = np.array([0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1])   # 날짜순 기록 12건
tr, te = y[:9], y[9:]                      # 앞 9건으로 배우고 뒤 3건으로 시험
print("훈련 가입률", round(tr.mean(), 2), "/ 시험 가입률", round(te.mean(), 2))
print("'모두 안 함' 시험 정확도", round((te == 0).mean(), 2))

### ❷ 넓혀 보기

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0)
X = np.arange(2000)[:, None] // 100        # 날짜순 2000건, 100건마다 다음 달(0~19)
y = rng.random(2000) < 0.05 + X[:, 0] / 50  # 달이 갈수록 가입이 늘어난다
mix = rng.random(2000) < 0.25              # 섞어서 25%를 시험으로
late = X[:, 0] >= 15                       # 마지막 5달을 시험으로
for k, b in [("섞어 나누기", mix), ("시간 분할", late)]:
    t = DecisionTreeClassifier(max_depth=3).fit(X[~b], y[~b])   # ~b = 시험이 아닌 행
    s = t.predict_proba(X[b])[:, 1]        # 시험 행의 가입 확률
    print(k, "AUC", round(roc_auc_score(y[b], s), 3))

## [실습 예제 3.3] 따라 하는 나무가 늘면 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
import numpy as np
ok = np.array([[1, 0, 1],                  # 고객 가: 나무1 ○, 나무2 ×, 나무3 ○
               [1, 1, 0],                  # 나
               [0, 1, 1],                  # 다
               [1, 1, 1]])                 # 라
print("나무마다 정확도", ok.mean(axis=0))
print("다수결 정확도", (ok.sum(axis=1) >= 2).mean())
same = np.repeat(ok[:, [0]], 3, axis=1)    # 세 그루가 모두 나무1과 같다면
print("똑같은 세 그루", (same.sum(axis=1) >= 2).mean())

### ❷ 넓혀 보기

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n, k = 20000, 25                           # 고객 수, 나무 수
lead = rng.random((n, 1)) < 0.6            # 첫 나무: 60%를 맞힘
for r in [0, 0.1, 0.3, 1]:                 # 첫 나무를 따라 할 확률
    copy = rng.random((n, k)) < r          # 따라 하는 자리
    own = rng.random((n, k)) < 0.6         # 스스로 판단해 60% 맞힘
    v = np.where(copy, lead, own)          # 따라 하면 첫 나무의 답, 아니면 제 답
    ok = (v.sum(axis=1) > k // 2).mean()   # 25그루 중 13그루 이상 맞으면 다수결 정답
    print("따라 할 확률", r, "→ 다수결 정확도", round(ok, 3))

## [실습 예제 3.4-1] AUC와 상위 30%를 함께 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
y = np.array([1, 0, 1, 0, 0, 1, 0, 0, 1, 0])   # 고객 10명의 가입 여부
a = np.linspace(0.9, 0, 10)                # 모델 A의 점수(위에서부터 차례로)
b = np.array([.8, .9, .5, .7, .6, .3, .4, .2, 0, .1])   # 모델 B의 점수
def top30(s):
    idx = np.argsort(-s)[:3]               # 점수 높은 위 3명(30%)의 번호
    return y[idx].sum() / y.sum()          # 그 안에서 만난 가입자의 비율
for k, s in [("A", a), ("B", b)]:
    print(k, "AUC", round(roc_auc_score(y, s), 3), "/ 상위 30%", top30(s))

## [실습 예제 3.4-2] 통화 시간이 만드는 AUC 🔵 코랩 실습

책(v8)에는 문제만 실렸고, 코드·결과·풀이는 이 노트북에서 돌려 봅니다.

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
cands = {"나이": "전", "잔고": "전", "마지막 통화 시간": "후",
         "지난 캠페인 결과": "전", "이번 캠페인 연락 횟수": "후(앞으로 걸 통화까지 셈)"}
for name, when in cands.items():           # 전화를 걸기 전에 아는 값인가?
    print(f"{name:12s} 걸기 {when:16s} →", "쓴다" if when == "전" else "빼거나 고친다")

### ❷ 넓혀 보기

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0)
age = rng.integers(20, 80, 5000)           # 나이: 걸기 전에 안다
y = rng.random(5000) < 0.05 + 0.003 * (age - 20)   # 나이가 많을수록 조금 더 가입
dur = rng.exponential(3, 5000) + 5 * y     # 통화 시간: 가입하면 5분쯤 길어진다(걸고 나서야 안다)
for k, x in [("나이", age), ("통화 시간", dur)]:
    print(k, "AUC", round(roc_auc_score(y, x), 3))   # 그 값 하나로 줄 세운 AUC

## [실습 예제 3.5-1] 두 중요도가 다를 때

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
import pandas as pd
q = pd.DataFrame({"변수": ["지난번 성공", "3월", "나이", "3월"],
                  "줄인 양": [0.020, 0.012, 0.006, 0.002]})   # 질문 넷이 줄인 불순도
imp = q.groupby("변수")["줄인 양"].sum()   # 변수마다 더하기
print((imp / imp.sum()).round(2).sort_values(ascending=False).to_dict())

### ❷ 넓혀 보기

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
rng = np.random.default_rng(0)
X = rng.integers(1, [13, 32], (3000, 2))   # 0열 달(1~12), 1열 날짜(1~31)
y = rng.random(3000) < X[:, 0] / 13        # 가입은 달에만 달려 있다
m = RandomForestClassifier(random_state=0).fit(X[::2], y[::2])   # 짝수 행으로 훈련
r = permutation_importance(m, X[1::2], y[1::2], random_state=0)  # 홀수 행으로 섞어 보기
print("불순도 중요도 [달, 날짜]", m.feature_importances_.round(3))
print("순열 중요도   [달, 날짜]", r.importances_mean.round(3))

## [실습 예제 6.2] 세 변수의 샤플리 값

### ❶ 앞 예제를 같은 숫자로 코드로 풀기

In [ ]:
v = {"": 0.10, "P": 0.40, "M": 0.25, "MP": 0.60}   # P 지난번 성공, M 3월
p_first = (v["P"] - v[""], v["MP"] - v["P"])         # 지난번을 먼저 알려 줄 때 (P 몫, M 몫)
m_first = (v["MP"] - v["M"], v["M"] - v[""])         # 3월을 먼저 알려 줄 때 (P 몫, M 몫)
phi_P = (p_first[0] + m_first[0]) / 2
phi_M = (p_first[1] + m_first[1]) / 2
print("지난번", round(phi_P, 3), "/ 3월", round(phi_M, 3), "/ 합", round(phi_P + phi_M, 3))

### ❷ 넓혀 보기

In [ ]:
from itertools import permutations
v = {"": .10, "P": .40, "M": .25, "A": .12,          # 아는 변수 묶음 → 확률
     "MP": .60, "AP": .43, "AM": .27, "AMP": .63}    # P 지난번 성공, M 3월, A 나이
key = lambda s: "".join(sorted(s))         # 묶음 이름을 정렬해 사전 열쇠로
phi = dict.fromkeys("PMA", 0.0)            # 변수마다 몫
for order in permutations("PMA"):          # 알려 주는 순서 6가지
    for i, f in enumerate(order):
        gain = v[key(order[:i + 1])] - v[key(order[:i])]   # f를 더 알려 줬을 때 오른 확률
        phi[f] += gain / 6                 # 여섯 순서의 평균
print({f: round(s, 3) for f, s in phi.items()}, "합", round(sum(phi.values()), 3))

# 05절 파이썬 실습: 은행 기록으로 나무와 숲 키우기

## [실습 예제 10-1] 데이터 불러오기와 연도 되살리기

### 단계 ❶

In [ ]:
URL = ("https://raw.githubusercontent.com/khatrideepti/"
       "Bank-Marketing-Data-Analysis/master/bank-full.csv")
bank = pd.read_csv(URL, sep=";")                 # 칸 구분 기호가 세미콜론
print(bank.shape)                                # (행 수, 열 수)
print(bank[["age", "job", "balance", "month", "duration", "poutcome", "y"]].head(3))

### 단계 ❷

In [ ]:
y = (bank["y"] == "yes").astype(int)             # 가입 1, 안 함 0
print("가입", y.sum(), "명, 비율", round(y.mean(), 3))
print("빈칸", bank.isna().sum().sum(),           # 비어 있는 칸의 총수
      "/ 연락 수단 unknown", (bank["contact"] == "unknown").sum())
num = bank.select_dtypes("number").columns       # 숫자로 된 열
print("숫자 열", len(num), "개 / 글자 열", bank.shape[1] - len(num), "개")
print("달 열의 값", list(bank["month"].unique()))   # 기록에 나오는 차례대로

### 단계 ❸

In [ ]:
MONTHS = ["jan", "feb", "mar", "apr", "may", "jun",
          "jul", "aug", "sep", "oct", "nov", "dec"]
m = bank["month"].map({k: i for i, k in enumerate(MONTHS)}).values   # 달 이름 → 0~11
new_year = np.r_[0, (m[1:] < m[:-1]).cumsum()]   # 달이 거꾸로 가면 1씩 늘어나는 번호
bank["year"] = 2008 + new_year
turn = np.flatnonzero(np.diff(new_year)) + 1     # 해가 바뀐 행의 번호
print("해가 바뀐 행", turn)
print(bank.loc[turn[0] - 1:turn[0], ["month", "day", "year"]])

### 단계 ❹

In [ ]:
by_year = bank.groupby("year").agg(              # 연도별로 묶어
    기록=("y", "size"), 가입률=("y", lambda s: (s == "yes").mean()))
print(by_year.round(3))
cut = int(len(bank) * 0.75)                      # 앞 75% / 뒤 25%의 경계 행
print("훈련 끝", bank["year"][cut - 1], bank["month"][cut - 1],
      "/ 시험 시작", bank["year"][cut], bank["month"][cut])
by_year["가입률"].plot.bar(color="#E04B3C", figsize=(5, 3))
plt.xlabel("연도"); plt.ylabel("가입률"); plt.xticks(rotation=0); plt.show()

### 직접 해 보기(정답)

In [ ]:
unk = bank.groupby("year")["contact"].apply(lambda s: (s == "unknown").mean())
print(unk.round(3).to_dict())

## [실습 예제 5.1] 작은 예제로 결정나무 한 걸음씩

### 단계 ❶

In [ ]:
import numpy as np
import pandas as pd
toy = pd.DataFrame({"age":  [25, 32, 38, 45, 52, 58, 63, 70],   # 나이
                    "prev": [0, 1, 0, 0, 1, 0, 1, 0],            # 지난번 가입
                    "y":    [0, 1, 0, 0, 1, 0, 1, 1]})           # 이번 가입
def gini(v):                                   # 0·1 배열의 지니 불순도
    p = np.mean(v)
    return 1 - p**2 - (1 - p)**2
print("고객", len(toy), "명, 가입", toy["y"].sum(), "명 → 뿌리 불순도", round(gini(toy["y"]), 3))

### 단계 ❷

In [ ]:
def best_split(d):                             # 가능한 질문을 모두 시험해 가장 좋은 것 고르기
    rows = []
    for col in ["age", "prev"]:
        vals = np.sort(d[col].unique())
        for c in (vals[:-1] + vals[1:]) / 2:   # 이웃한 값의 가운데 = 문턱 후보
            L, R = d["y"][d[col] <= c], d["y"][d[col] > c]
            after = (len(L) * gini(L) + len(R) * gini(R)) / len(d)
            rows.append((col, c, round(gini(d["y"]) - after, 3)))   # (변수, 문턱, 줄어든 불순도)
    return pd.DataFrame(rows, columns=["변수", "문턱", "질문의 값"]).sort_values("질문의 값", ascending=False)
cand = best_split(toy)
print(cand.head(4).to_string(index=False))

### 단계 ❸

In [ ]:
q = cand.iloc[0]                               # 1등 질문
left, right = toy[toy[q["변수"]] <= q["문턱"]], toy[toy[q["변수"]] > q["문턱"]]
print("첫 질문:", q["변수"], "<=", q["문턱"])
print("왼쪽", len(left), "명 가입률", round(left["y"].mean(), 2), "/ 오른쪽", len(right), "명 가입률", round(right["y"].mean(), 2))
print("왼쪽에서 다시 고른 질문:", best_split(left).iloc[0].to_dict())

### 단계 ❹

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
t = DecisionTreeClassifier(max_depth=2, random_state=0).fit(toy[["age", "prev"]], toy["y"])
print(export_text(t, feature_names=["age", "prev"]))

## [실습 예제 10-2] 첫 질문 고르기: 지니 불순도 손 계산

### 단계 ❶

In [ ]:
def gini(p):                                     # 가입 비율이 p인 무리의 지니 불순도(식 10-1)
    return 1 - p**2 - (1 - p)**2
root = gini(y.mean())                            # 나누기 전, 뿌리의 불순도
print("가입 비율", round(y.mean(), 3), "→ 뿌리 불순도", round(root, 4))

### 단계 ❷

In [ ]:
yes = bank["poutcome"] == "success"              # 질문: 지난 캠페인 성공? (행마다 True/False)
a, b = y[yes], y[~yes]                           # 예 / 아니요 두 무리
print("예", len(a), "명 가입률", round(a.mean(), 3), "불순도", round(gini(a.mean()), 4))
print("아니요", len(b), "명 가입률", round(b.mean(), 3), "불순도", round(gini(b.mean()), 4))
after = (len(a) * gini(a.mean()) + len(b) * gini(b.mean())) / len(y)   # 가중 평균(식 10-2)
print("나눈 뒤", round(after, 4), "/ 질문의 값", round(root - after, 4))

### 단계 ❸

In [ ]:
def gain(yes):                                   # 질문 하나의 값 = 줄어든 불순도
    a, b = y[yes], y[~yes]
    return root - (len(a) * gini(a.mean()) + len(b) * gini(b.mean())) / len(y)
questions = {"지난 캠페인 성공?": bank["poutcome"] == "success",
             "60세 초과?": bank["age"] > 60,
             "주택대출 있음?": bank["housing"] == "yes"}
vals = pd.Series({q: gain(yes) for q, yes in questions.items()})
print(vals.round(4).sort_values(ascending=False))
vals.sort_values().plot.barh(color="#E04B3C", figsize=(5, 2.4))
plt.xlabel("지니 불순도 감소"); plt.show()

### 직접 해 보기(정답)

In [ ]:
print("잔고 1,000유로 초과?", round(gain(bank["balance"] > 1000), 4))

## [실습 예제 10-3] 시간 순서로 나누고 깊이 3의 나무 키우기

### 단계 ❶

In [ ]:
X = pd.get_dummies(bank.drop(columns=["y", "year", "duration"]), dtype=int)   # 답·연도·통화 시간 빼고 펼치기
print("열", bank.shape[1], "개 → 특징", X.shape[1], "개")
print([c for c in X.columns if c.startswith("month_")][:4], "…")

### 단계 ❷

In [ ]:
Xtr, Xte = X.iloc[:cut], X.iloc[cut:]            # 앞 75%로 배우고 뒤 25%로 시험
ytr, yte = y.iloc[:cut], y.iloc[cut:]
print("훈련", len(Xtr), "행 / 시험", len(Xte), "행")
print("가입률: 훈련", round(ytr.mean(), 3), "/ 시험", round(yte.mean(), 3))
print("모두 '안 함'으로 찍은 정확도", round(1 - yte.mean(), 3))

### 단계 ❸

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import roc_auc_score
def auc(model, Xs, ys):                          # 확률로 줄 세우는 능력(이 장의 채점 함수)
    return roc_auc_score(ys, model.predict_proba(Xs)[:, 1])
t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)   # 질문 세 번까지
print(export_text(t3, feature_names=list(X.columns), max_depth=0))      # 첫 질문만 글자로
print("정확도", round(t3.score(Xte, yte), 3), "/ AUC", round(auc(t3, Xte, yte), 3))
from sklearn.tree import plot_tree
plt.figure(figsize=(10, 4))
plot_tree(t3, max_depth=2, feature_names=list(X.columns), class_names=["안 함", "가입"], filled=True, impurity=False, proportion=True, fontsize=9)
plt.show()

### 단계 ❹

In [ ]:
p3 = t3.predict_proba(Xte.iloc[:3])[:, 1]        # 시험 기간 첫 고객 셋의 가입 확률
print("확률", p3.round(3), "/ 0.5 문턱 판정", t3.predict(Xte.iloc[:3]))
print("잎 번호", t3.apply(Xte.iloc[:3]))         # 각 고객이 닿은 잎

### 직접 해 보기(정답)

In [ ]:
t2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xtr, ytr)
print("깊이 2: 정확도", round(t2.score(Xte, yte), 3), "/ AUC", round(auc(t2, Xte, yte), 3))

## [실습 예제 10-4] 끝까지 자란 나무와 과적합

### 단계 ❶

In [ ]:
full = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)   # 깊이 제한 없음
print("깊이", full.get_depth(), "/ 잎", full.get_n_leaves())
print("훈련 정확도", round(full.score(Xtr, ytr), 3),
      "/ 시험 정확도", round(full.score(Xte, yte), 3))

### 단계 ❷

In [ ]:
for depth in [3, 8, 20]:                         # 깊이 세 개만 먼저
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xtr, ytr)
    print(f"깊이 {depth}: 훈련 AUC {auc(t, Xtr, ytr):.3f}, 시험 AUC {auc(t, Xte, yte):.3f}")

### 단계 ❸

In [ ]:
curve = {}
for d in range(1, 26):                           # 깊이 1~25를 모두
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    curve[d] = (auc(t, Xtr, ytr), auc(t, Xte, yte))   # (훈련, 시험)
best = max(curve, key=lambda d: curve[d][1])     # 시험 AUC가 가장 높은 깊이
print("가장 좋은 깊이", best, "시험 AUC", round(curve[best][1], 3))
plt.figure(figsize=(6, 3))
plt.plot(list(curve), [a for a, b in curve.values()], color="#9AA0A6", marker="o", ms=3, label="훈련")
plt.plot(list(curve), [b for a, b in curve.values()], color="#E04B3C", marker="o", ms=3, label="시험")
plt.xlabel("나무의 깊이"); plt.ylabel("AUC"); plt.legend(); plt.show()

### 직접 해 보기(정답)

In [ ]:
t50 = DecisionTreeClassifier(min_samples_leaf=50, random_state=0).fit(Xtr, ytr)
print("잎 최소 50명: 깊이", t50.get_depth(), "잎", t50.get_n_leaves(), "시험 AUC", round(auc(t50, Xte, yte), 3))

## [실습 예제 10-5] 교차검증으로 잎의 크기 고르기

### 단계 ❶

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(5, shuffle=True, random_state=0)   # 훈련 기록을 다섯 조각으로
for i, (fit_idx, val_idx) in enumerate(cv.split(Xtr, ytr), 1):
    print(i, "조각: 배우기", len(fit_idx), "/ 검증", len(val_idx),
          "/ 검증 가입률", round(ytr.iloc[val_idx].mean(), 3))

### 단계 ❷

In [ ]:
cvres = {}
for n in [10, 50, 100, 200, 400]:               # 잎마다 최소 인원 후보
    s = cross_val_score(DecisionTreeClassifier(min_samples_leaf=n, random_state=0),
                        Xtr, ytr, cv=cv, scoring="roc_auc")   # 다섯 번의 검증 AUC
    cvres[f"{n}명"] = s
    print(f"잎마다 최소 {n}명: 검증 AUC {s.mean():.3f} (±{s.std():.3f})")
plt.figure(figsize=(5, 2.8))
plt.boxplot(list(cvres.values()), tick_labels=list(cvres.keys()), vert=False)
plt.xlabel("다섯 조각의 검증 AUC"); plt.ylabel("잎의 최소 인원"); plt.show()

### 단계 ❸

In [ ]:
pruned = DecisionTreeClassifier(min_samples_leaf=100, random_state=0).fit(Xtr, ytr)   # 고른 값으로 훈련 전체에 다시
print("잎 최소 100명: 잎", pruned.get_n_leaves(), "/ 시험 AUC", round(auc(pruned, Xte, yte), 3))

### 직접 해 보기(정답)

In [ ]:
from sklearn.model_selection import GridSearchCV
g = GridSearchCV(DecisionTreeClassifier(random_state=0),
                 {"min_samples_leaf": [50, 100, 200], "max_depth": [5, 10, None]},
                 cv=cv, scoring="roc_auc").fit(Xtr, ytr)
print(g.best_params_, round(g.best_score_, 3))

## [실습 예제 5.2] 작은 예제로 랜덤 포레스트 한 걸음씩

### 단계 ❶

In [ ]:
rng = np.random.default_rng(1)
bags = [rng.integers(0, 8, 8) for _ in range(3)]   # 고객 번호 0~7을 중복 허락해 8번씩, 세 묶음
for i, b in enumerate(bags, 1):
    print(f"묶음 {i}: 고객 {sorted(b.tolist())} / 빠진 고객 {sorted(set(range(8)) - set(b.tolist()))}")

### 단계 ❷

In [ ]:
stumps = []
for i, b in enumerate(bags, 1):
    d = toy.iloc[b]                             # 그 묶음의 고객들(중복 포함)
    s = DecisionTreeClassifier(max_depth=1, random_state=0).fit(d[["age", "prev"]], d["y"])
    stumps.append(s)
    f = ["age", "prev"][s.tree_.feature[0]]     # 그 나무의 첫 질문 변수
    print(f"나무 {i}: 질문 {f} <= {s.tree_.threshold[0]:.1f}")

### 단계 ❸

In [ ]:
new = pd.DataFrame({"age": [30, 66], "prev": [0, 0]})   # 새 고객 둘
P = np.array([s.predict_proba(new)[:, 1] for s in stumps])   # (나무 3, 고객 2) 확률표
print(pd.DataFrame(P.round(2), index=["나무1", "나무2", "나무3"], columns=["30세", "66세"]))
print("숲(평균)", P.mean(axis=0).round(2))

### 단계 ❹

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_toy = RandomForestClassifier(n_estimators=3, max_depth=1, max_features=1, random_state=0)
rf_toy.fit(toy[["age", "prev"]], toy["y"])
each = np.array([e.predict_proba(new.values)[:, 1] for e in rf_toy.estimators_])
print("나무마다\n", each.round(2))
print("나무 평균", each.mean(axis=0).round(2), "= predict_proba", rf_toy.predict_proba(new)[:, 1].round(2))

## [실습 예제 10-6] 다르게 틀려야 모은 보람이 있다

### 단계 ❶

In [ ]:
from sklearn.ensemble import RandomForestClassifier
def forest(**kw):                                # 나머지 설정은 셋 다 같게
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                  random_state=0, n_jobs=-1, **kw).fit(Xtr, ytr)
kinds = {"똑같은 나무": forest(max_features=None, bootstrap=False),   # 같은 데이터·모든 변수
         "배깅": forest(max_features=None),                          # 다시 뽑은 데이터·모든 변수
         "랜덤 포레스트": forest(max_features="sqrt")}               # 다시 뽑은 데이터·변수 √50 ≒ 7개
print({k: len(f.estimators_) for k, f in kinds.items()})

### 단계 ❷

In [ ]:
bars = {}
for name, f in kinds.items():
    one = [roc_auc_score(yte, t.predict_proba(Xte.values)[:, 1])
           for t in f.estimators_[:30]]          # 숲 속 나무를 한 그루씩 채점(30그루)
    first = {t.tree_.feature[0] for t in f.estimators_}   # 나무마다 첫 질문에 쓴 변수
    print(f"{name}: 나무 하나 {np.mean(one):.3f} / 숲 {auc(f, Xte, yte):.3f}"
          f" / 첫 질문 {len(first)}가지")
    bars[name] = (np.mean(one), auc(f, Xte, yte))
bt = pd.DataFrame(bars, index=["나무 한 그루", "숲 300그루"]).T
bt.plot.bar(color=["#9AA0A6", "#E04B3C"], figsize=(5, 3), rot=0, ylim=(0.5, 0.75))
plt.ylabel("시험 AUC"); plt.show()

### 단계 ❸

In [ ]:
rf = kinds["랜덤 포레스트"]
probs = np.array([t.predict_proba(Xte.values)[:, 1] for t in rf.estimators_])   # (300그루, 고객) 확률표
for k in [1, 10, 30, 300]:                       # 앞의 k그루만 평균하면
    print(f"{k}그루 평균: AUC {roc_auc_score(yte, probs[:k].mean(axis=0)):.3f}")
ks = list(range(1, 31)) + list(range(40, 301, 20))
plt.figure(figsize=(6, 2.8))
plt.plot(ks, [roc_auc_score(yte, probs[:k].mean(axis=0)) for k in ks], color="#E04B3C")
plt.xscale("log"); plt.xlabel("평균한 나무의 수(로그 눈금)"); plt.ylabel("시험 AUC"); plt.show()

### 직접 해 보기(정답)

In [ ]:
imp_rf = pd.Series(rf.feature_importances_, X.columns).nlargest(3).round(3)
print(imp_rf.to_dict())

## [실습 예제 5.3] 작은 예제로 그레이디언트 부스팅 한 걸음씩

### 단계 ❶

In [ ]:
x = toy[["age"]].values                         # 나이 하나로
yv = toy["y"].values.astype(float)
F = np.full(8, yv.mean())                       # 출발: 모두 가입 비율 0.5
print("출발 예측", F.round(3), "/ 잔차", (yv - F).round(3))

### 단계 ❷

In [ ]:
from sklearn.tree import DecisionTreeRegressor
for m in range(1, 4):
    r = yv - F                                  # 잔차
    h = DecisionTreeRegressor(max_depth=1).fit(x, r)   # 잔차를 맞히는 질문 하나짜리 나무
    F = F + 0.5 * h.predict(x)                  # 학습률 0.5만큼 더하기
    print(f"{m}그루: 질문 age <= {h.tree_.threshold[0]:.1f} / 잔차 제곱합 {((yv - F) ** 2).sum():.3f}")
print("손으로 한 예측", F.round(3))

### 단계 ❸

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
g = GradientBoostingRegressor(n_estimators=3, learning_rate=0.5, max_depth=1).fit(x, yv)
print("사이킷런 예측 ", g.predict(x).round(3))

## [실습 예제 10-7] 부스팅과 로지스틱 회귀 견주기

### 단계 ❶

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
def boost(rate, Xa=Xtr, ya=ytr):                 # 학습률과 훈련 데이터만 바꿔 쓰는 부스팅
    return GradientBoostingClassifier(n_estimators=300, learning_rate=rate,
                                      max_depth=3, subsample=0.8,
                                      random_state=0).fit(Xa, ya)
gb, fast = boost(0.05), boost(1.0)
slow_auc = [roc_auc_score(yte, p[:, 1]) for p in gb.staged_predict_proba(Xte)]     # 1그루, 2그루, … 300그루일 때
fast_auc = [roc_auc_score(yte, p[:, 1]) for p in fast.staged_predict_proba(Xte)]
for k in [1, 50, 100, 300]:
    print(f"{k}그루: 학습률 0.05 → {slow_auc[k - 1]:.3f}, 1.0 → {fast_auc[k - 1]:.3f}")
print("0.05의 최고", round(max(slow_auc), 3), "(", int(np.argmax(slow_auc)) + 1, "그루)")
plt.figure(figsize=(6, 3))
plt.plot(range(1, 301), slow_auc, color="#E04B3C", label="학습률 0.05")
plt.plot(range(1, 301), fast_auc, color="#3F7CC0", label="학습률 1.0")
plt.xlabel("더한 나무의 수"); plt.ylabel("시험 AUC"); plt.legend(); plt.show()

### 단계 ❷

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)   # 표준화 → 로지스틱 회귀
print("가지친 나무: 잎", pruned.get_n_leaves(), "/ 로지스틱 회귀 계수", lr[-1].coef_.shape[1], "개")

### 단계 ❸

In [ ]:
board = {"깊이 3 나무": t3, "가지친 나무": pruned, "랜덤 포레스트": rf,
         "부스팅": gb, "로지스틱 회귀": lr}
plt.figure(figsize=(5, 3.6))
for name, model in board.items():
    p = model.predict_proba(Xte)[:, 1]                         # 시험 고객의 가입 확률
    found = yte.values[np.argsort(-p)].cumsum() / yte.sum()   # 확률 높은 순으로 걸 때 만난 가입자 비율
    print(f"{name}: AUC {auc(model, Xte, yte):.3f}, 상위 10% {found[len(p) // 10 - 1]:.3f},"
          f" 상위 30% {found[len(p) * 3 // 10 - 1]:.3f}")
    if name in ("깊이 3 나무", "랜덤 포레스트", "로지스틱 회귀"): plt.plot(np.arange(1, len(p) + 1) / len(p), found, label=name)
plt.plot([0, 1], [0, 1], "k--", lw=1, label="아무에게나")
plt.xlabel("전화한 고객의 비율"); plt.ylabel("만난 가입자의 비율"); plt.legend(); plt.show()

### 단계 ❹

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score
p = lr.predict_proba(Xte)[:, 1]
call = (p >= np.quantile(p, 0.7)).astype(int)    # 확률 상위 30%에게 전화(1)
print(confusion_matrix(yte, call))               # [[안함·안걸기, 안함·전화], [가입·안걸기, 가입·전화]]
print("정밀도", round(precision_score(yte, call), 3),
      "/ 재현율", round(recall_score(yte, call), 3))
print("문턱 0.5로 전화할 사람", (p >= 0.5).sum(), "명")

### 직접 해 보기(정답)

In [ ]:
for q in [0.9, 0.5]:                             # 상위 10%, 상위 50%
    c = (p >= np.quantile(p, q)).astype(int)
    print(f"상위 {1 - q:.0%}: 정밀도 {precision_score(yte, c):.3f}, 재현율 {recall_score(yte, c):.3f}")

## [실습 예제 5.4] 작은 예제로 순열 중요도 한 걸음씩

### 단계 ❶

In [ ]:
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0)
X4 = pd.DataFrame({"prev": rng.integers(0, 2, 400),      # 가입과 관계있는 변수
                   "day": rng.integers(1, 32, 400)})      # 관계없는 변수(날짜)
y4 = (rng.random(400) < 0.15 + 0.5 * X4["prev"]).astype(int)
m4 = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X4[:300], y4[:300])
Xv, yvv = X4[300:], y4[300:]                    # 검사용 100명
base = roc_auc_score(yvv, m4.predict_proba(Xv)[:, 1])
print("섞기 전 AUC", round(base, 3))

### 단계 ❷

In [ ]:
for col in ["prev", "day"]:
    Xs = Xv.copy()
    Xs[col] = rng.permutation(Xs[col].values)  # 그 열만 고객끼리 뒤섞기
    auc_s = roc_auc_score(yvv, m4.predict_proba(Xs)[:, 1])
    print(f"{col} 섞으면 AUC {auc_s:.3f} → 떨어진 양 {base - auc_s:.3f}")

### 단계 ❸

In [ ]:
from sklearn.inspection import permutation_importance
pi4 = permutation_importance(m4, Xv, yvv, scoring="roc_auc", n_repeats=10, random_state=0)
print("순열 중요도", {c: float(v) for c, v in zip(X4.columns, pi4.importances_mean.round(3))})
print("불순도 중요도", {c: float(v) for c, v in zip(X4.columns, m4.feature_importances_.round(3))})

## [실습 예제 10-8] 통화 시간과 두 가지 중요도

### 단계 ❶

In [ ]:
Xd = pd.get_dummies(bank.drop(columns=["y", "year"]), dtype=int)   # 통화 시간을 넣은 특징
gbd = boost(0.05, Xd.iloc[:cut], ytr)
print("통화 시간을 넣으면: 시험 AUC", round(auc(gbd, Xd.iloc[cut:], yte), 3), "(빼면", round(auc(gb, Xte, yte), 3), ")")
mins = pd.cut(bank["duration"] / 60, [0, 1, 3, 5, 10, 100], right=False)   # 통화 시간(분)을 구간으로
print(y.groupby(mins, observed=True).mean().round(3).to_string())
y.groupby(mins, observed=True).mean().plot.bar(color="#E04B3C", figsize=(5, 2.8), rot=0)
plt.xlabel("통화 시간(분)"); plt.ylabel("가입률"); plt.show()

### 단계 ❷

In [ ]:
imp = pd.Series(gb.feature_importances_, X.columns)      # 불순도 중요도(훈련 때 줄인 불순도의 몫)
print("불순도 중요도", imp.nlargest(3).round(3).to_dict())
imp.sort_values().tail(8).plot.barh(color="#9AA0A6", figsize=(5, 3.5))
plt.xlabel("불순도 중요도"); plt.show()

### 단계 ❸

In [ ]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(gb, Xte, yte, scoring="roc_auc", n_repeats=5,
                            random_state=0, n_jobs=-1)   # 시험 데이터에서 열을 하나씩 섞어 보기(5번)
perm = pd.Series(pi.importances_mean, X.columns)         # 순열 중요도
print("순열 중요도", perm.nlargest(3).round(4).to_dict())
mar = bank[bank["month"] == "mar"]
print("3월 기록", len(mar), "건, 연도", sorted(mar["year"].unique().tolist()))
perm.sort_values().tail(8).plot.barh(color="#E04B3C", figsize=(5, 3.5))
plt.xlabel("섞었을 때 떨어진 시험 AUC"); plt.show()

### 직접 해 보기(정답)

In [ ]:
print(bank.groupby("year")["duration"].median().to_dict())

## [실습 예제 10-9] 나누는 방법을 바꾸면

### 단계 ❶

In [ ]:
from sklearn.model_selection import train_test_split
Rtr, Rte, rtr, rte = train_test_split(X, y, test_size=0.25, stratify=y,
                                      random_state=42)   # 세 해를 섞어 무작위로
mixed = DecisionTreeClassifier(min_samples_leaf=100, random_state=0).fit(Rtr, rtr)
lrr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Rtr, rtr)
gbr = boost(0.05, Rtr, rtr)
print("무작위 분할: 가지친 나무", round(auc(mixed, Rte, rte), 3),
      "/ 부스팅", round(auc(gbr, Rte, rte), 3),
      "/ 로지스틱", round(auc(lrr, Rte, rte), 3))

### 단계 ❷

In [ ]:
c8 = int(len(X) * 0.8)                           # 자르는 곳을 80%로 옮기면
lr8 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
lr8.fit(X.iloc[:c8], y.iloc[:c8])
gb8 = boost(0.05, X.iloc[:c8], y.iloc[:c8])
print("80%에서 자르면: 부스팅", round(auc(gb8, X.iloc[c8:], y.iloc[c8:]), 3),
      "/ 로지스틱", round(auc(lr8, X.iloc[c8:], y.iloc[c8:]), 3),
      "/ 경계", bank["year"][c8], bank["month"][c8])
cmp = pd.DataFrame({"무작위 분할": [auc(mixed, Rte, rte), auc(gbr, Rte, rte), auc(lrr, Rte, rte)],
                    "시간 분할(75%)": [auc(pruned, Xte, yte), auc(gb, Xte, yte), auc(lr, Xte, yte)]},
                   index=["가지친 나무", "부스팅", "로지스틱"])
cmp.plot.bar(color=["#9AA0A6", "#E04B3C"], figsize=(5, 3), rot=0, ylim=(0.5, 0.85))
plt.ylabel("시험 AUC"); plt.show()

### 단계 ❸

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
ts = TimeSeriesSplit(n_splits=4)                 # 앞으로 배우고 바로 뒤로 검증 × 4
for i, (a_, b_) in enumerate(ts.split(X), 1):
    print(i, "번째: 배우기", len(a_), "행 / 검증", bank["year"][b_[0]], bank["month"][b_[0]], "~", len(b_), "행")
for name, mdl in [("가지친 나무", pruned), ("로지스틱", lr)]:
    s = cross_val_score(mdl, X, y, cv=ts, scoring="roc_auc")
    print(name, s.round(3), "평균", round(s.mean(), 3))

### 직접 해 보기(정답)

In [ ]:
s = cross_val_score(DecisionTreeClassifier(min_samples_leaf=100, random_state=0),
                    X, y, cv=StratifiedKFold(4, shuffle=True, random_state=0), scoring="roc_auc")
print("섞은 4조각 교차검증", s.round(3), "평균", round(s.mean(), 3))